# Experiment 10: Unsupervised Cyber Network Traffic Anomaly Detection

## 1. Overview & Research Objectives
This experiment benchmarks **Unsupervised Anomaly / Novelty Detection** on **Cyber Network Traffic**:
- **Baseline Calibration**: Models are trained strictly on legitimate **`Benign`** Wi-Fi/TCP/UDP packet streams (28 leakage-free packet metrics: frame lengths, protocol flags, port behaviors, sequence numbers).
- **Novelty Detection**: Testing against unseen Benign packets + all 4 attack vectors (`DoS`, `Evil_Twin`, `FDI`, `Replay`).
- **Target Question**: Can cyber anomaly detectors detect volumetric flood attacks (`DoS`) and rogue access points (`Evil_Twin`) purely from normal packet baselines?


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem

from utils.data_loader import load_cyber_dataset, get_novelty_detection_split
from utils.unsupervised_metrics import evaluate_anomaly_detector, measure_inference_speed, get_model_size_kb


## 2. Cyber Data Loading & Novelty Split


In [ ]:
X_c, y_c, feature_names = load_cyber_dataset("../Cyber_UAV_Dataset.csv")
print(f"[*] Total Cyber Dataset: {X_c.shape[0]} samples, {X_c.shape[1]} network features")

X_tr, X_te, y_te_bin, y_te_multi = get_novelty_detection_split(X_c, y_c, benign_train_ratio=0.7)
print(f"[*] Training Baseline (Benign only): {X_tr.shape[0]} packets")
print(f"[*] Testing Set (Unseen Benign + Attacks): {X_te.shape[0]} packets")

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)


## 3. Cyber Models: Isolation Forest & Nystroem OC-SVM


In [ ]:
# 1. Isolation Forest
iforest_c = IsolationForest(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_s)
tr_scores_c = -iforest_c.score_samples(X_tr_s)
te_scores_c = -iforest_c.score_samples(X_te_s)
lat_if = measure_inference_speed(lambda x: -iforest_c.score_samples(x), X_te_s)
m_if_opt, _, _ = evaluate_anomaly_detector(te_scores_c, y_te_bin, y_te_multi, "Cyber iForest (Best F1)", "Cyber", lat_if)

tau_5 = np.percentile(tr_scores_c, 95)
m_if_5, _, _ = evaluate_anomaly_detector(te_scores_c, y_te_bin, y_te_multi, "Cyber iForest (5% FAR)", "Cyber", lat_if, threshold=tau_5)

# 2. Scalable Nystroem Kernel OC-SVM
nystroem = Nystroem(n_components=50, random_state=42)
X_tr_nys = nystroem.fit_transform(X_tr_s)
X_te_nys = nystroem.transform(X_te_s)
ocsvm_c = OneClassSVM(nu=0.05, kernel='linear').fit(X_tr_nys)
te_scores_svm = -ocsvm_c.decision_function(X_te_nys)
lat_svm = measure_inference_speed(lambda x: -ocsvm_c.decision_function(nystroem.transform(x)), X_te_s)
m_svm, _, _ = evaluate_anomaly_detector(te_scores_svm, y_te_bin, y_te_multi, "Cyber OC-SVM (Nystroem)", "Cyber", lat_svm)

df_cyb_summary = pd.DataFrame([m_if_opt, m_if_5, m_svm])
df_cyb_summary


## 4. Anomaly Score Distribution Across Packet Types


In [ ]:
plt.figure(figsize=(10, 6))
df_plot = pd.DataFrame({'Score': te_scores_c, 'Attack': y_te_multi})
sns.kdeplot(data=df_plot, x='Score', hue='Attack', common_norm=False, fill=True, alpha=0.3, palette='tab10')
plt.axvline(tau_5, color='red', linestyle='--', label=f'5% FAR Threshold ({tau_5:.3f})')
plt.title("Cyber Isolation Forest: Anomaly Score Distribution by Packet Stream", fontsize=13, fontweight='bold')
plt.xlabel("Anomaly Score (Higher = More Anomalous)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Key Empirical Insights
1. **Network Attack Sensitivity**: Unlike physical sensors, cyber models capture **DoS packet flooding and Evil_Twin rogue beacon bursts** with high anomaly scores.
2. **Complementarity**: The cyber anomaly profile is orthogonal to physical kinematics, establishing the foundation for cross-domain multimodal fusion.
